# Notebook 2: 概率统计与数据分析

本 Notebook 整合概率统计概念与 Python 实践，使用 SciPy 和 Pandas 进行数据分析。

## 1. 导入库

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置样式
sns.set_style('whitegrid')
np.random.seed(42)

print("库版本:")
print(f"  NumPy: {np.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  SciPy: {stats.__version__ if hasattr(stats, '__version__') else 'N/A'}")

## 2. 常见概率分布

In [ ]:
# 二项分布
n, p = 10, 0.5
binom_dist = stats.binom(n, p)

x_binom = np.arange(0, 11)
pmf_binom = binom_dist.pmf(x_binom)

plt.figure(figsize=(10, 5))
plt.bar(x_binom, pmf_binom, alpha=0.7, color='skyblue', edgecolor='black')
plt.xlabel('成功次数 k')
plt.ylabel('P(X=k)')
plt.title(f'二项分布 B({n}, {p})')
plt.grid(True, alpha=0.3)
plt.savefig('/home/admin/.openclaw/workspace/DataScience_Math_Python/notebooks/binomial_dist.png', dpi=150)
plt.show()

print(f"期望值 E[X] = {binom_dist.mean():.2f} (公式：np = {n*p})")
print(f"方差 Var(X) = {binom_dist.var():.2f} (公式：np(1-p) = {n*p*(1-p)})")

In [ ]:
# 正态分布
mu, sigma = 0, 1
norm_dist = stats.norm(mu, sigma)

x_norm = np.linspace(-4, 4, 100)
pdf_norm = norm_dist.pdf(x_norm)

plt.figure(figsize=(10, 5))
plt.plot(x_norm, pdf_norm, 'b-', linewidth=2, label='N(0,1)')
plt.fill_between(x_norm, 0, pdf_norm, where=(x_norm>=-1)&(x_norm<=1), alpha=0.3, label='68%')
plt.fill_between(x_norm, 0, pdf_norm, where=(x_norm>=-2)&(x_norm<=2), alpha=0.2, label='95%')
plt.xlabel('x')
plt.ylabel('概率密度')
plt.title('标准正态分布')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('/home/admin/.openclaw/workspace/DataScience_Math_Python/notebooks/normal_dist.png', dpi=150)
plt.show()

# 68-95-99.7 规则验证
print("68-95-99.7 规则:")
print(f"  P(-1 ≤ Z ≤ 1) = {norm_dist.cdf(1) - norm_dist.cdf(-1):.4f}")
print(f"  P(-2 ≤ Z ≤ 2) = {norm_dist.cdf(2) - norm_dist.cdf(-2):.4f}")
print(f"  P(-3 ≤ Z ≤ 3) = {norm_dist.cdf(3) - norm_dist.cdf(-3):.4f}")

## 3. 描述统计

In [ ]:
# 生成示例数据
data = np.random.normal(loc=75, scale=10, size=1000)

# 创建 DataFrame
df = pd.DataFrame({
    '成绩': data,
    '是否及格': data >= 60
})

print("数据概览:")
print(df.head())

print("\n描述统计:")
print(df['成绩'].describe())

In [ ]:
# 可视化分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 直方图 + KDE
sns.histplot(df['成绩'], kde=True, bins=30, ax=axes[0], color='skyblue')
axes[0].axvline(df['成绩'].mean(), color='red', linestyle='--', label=f'均值={df["成绩"].mean():.2f}')
axes[0].axvline(df['成绩'].median(), color='green', linestyle='--', label=f'中位数={df["成绩"].median():.2f}')
axes[0].legend()
axes[0].set_title('成绩分布')

# 箱线图
sns.boxplot(y=df['成绩'], ax=axes[1], color='lightgreen')
axes[1].set_title('成绩箱线图')
axes[1].axhline(df['成绩'].mean(), color='red', linestyle='--', label='均值')
axes[1].legend()

plt.tight_layout()
plt.savefig('/home/admin/.openclaw/workspace/DataScience_Math_Python/notebooks/descriptive_stats.png', dpi=150)
plt.show()

print(f"\n集中趋势:")
print(f"  均值：{df['成绩'].mean():.4f}")
print(f"  中位数：{df['成绩'].median():.4f}")
print(f"  众数：{stats.mode(df['成绩'].astype(int))[0][0]}")

print(f"\n离散程度:")
print(f"  方差：{df['成绩'].var():.4f}")
print(f"  标准差：{df['成绩'].std():.4f}")
print(f"  IQR: {df['成绩'].quantile(0.75) - df['成绩'].quantile(0.25):.4f}")

## 4. 贝叶斯定理应用

In [ ]:
# 疾病检测问题
# P(患病) = 0.001 (患病率 0.1%)
# P(阳性 | 患病) = 0.99 (灵敏度)
# P(阴性 | 健康) = 0.99 (特异度)

p_disease = 0.001
p_positive_given_disease = 0.99
p_negative_given_healthy = 0.99

# 计算 P(阳性)
p_positive_given_healthy = 1 - p_negative_given_healthy
p_healthy = 1 - p_disease

p_positive = (p_positive_given_disease * p_disease + 
              p_positive_given_healthy * p_healthy)

# 贝叶斯定理：P(患病 | 阳性)
p_disease_given_positive = (p_positive_given_disease * p_disease) / p_positive

print("贝叶斯定理 - 疾病检测问题:")
print(f"\n已知条件:")
print(f"  患病率 P(患病) = {p_disease}")
print(f"  灵敏度 P(阳性 | 患病) = {p_positive_given_disease}")
print(f"  特异度 P(阴性 | 健康) = {p_negative_given_healthy}")

print(f"\n计算结果:")
print(f"  P(阳性) = {p_positive:.6f}")
print(f"  P(患病 | 阳性) = {p_disease_given_positive:.6f}")
print(f"  ≈ {p_disease_given_positive * 100:.2f}%")

print(f"\n解释：即使检测准确率高达 99%，")
print(f"由于患病率很低，阳性结果中真正患病的比例仍然很低。")

## 5. 假设检验

In [ ]:
# 单样本 t 检验
# 某公司声称产品平均寿命为 1000 小时
# 抽样 25 个产品，测得平均寿命 980 小时，标准差 50 小时

np.random.seed(42)
sample_data = np.random.normal(loc=980, scale=50, size=25)

mu_0 = 1000  # 零假设的均值

# 执行 t 检验
t_stat, p_value = stats.ttest_1samp(sample_data, mu_0)

print("产品寿命假设检验:")
print(f"  样本均值 = {sample_data.mean():.2f} 小时")
print(f"  样本标准差 = {sample_data.std(ddof=1):.2f} 小时")
print(f"  样本大小 = {len(sample_data)}")
print(f"  零假设 H0: μ = {mu_0}")
print(f"  备择假设 H1: μ ≠ {mu_0}")
print(f"\n  t 统计量 = {t_stat:.4f}")
print(f"  p 值 = {p_value:.4f}")

alpha = 0.05
print(f"\n  显著性水平 α = {alpha}")
if p_value < alpha:
    print(f"  结论：p < α，拒绝 H0，产品寿命与声称值有显著差异")
else:
    print(f"  结论：p ≥ α，不拒绝 H0，没有足够证据否定声称")

In [ ]:
# 双样本 t 检验
# 比较两种教学方法的效果

np.random.seed(42)
method_a = np.random.normal(75, 10, 30)  # 方法 A
method_b = np.random.normal(80, 10, 30)  # 方法 B

t_stat_2, p_value_2 = stats.ttest_ind(method_a, method_b)

print("\n双样本 t 检验 - 教学方法比较:")
print(f"  方法 A 平均分：{method_a.mean():.2f}")
print(f"  方法 B 平均分：{method_b.mean():.2f}")
print(f"  t 统计量：{t_stat_2:.4f}")
print(f"  p 值：{p_value_2:.4f}")

if p_value_2 < 0.05:
    print(f"  结论：两种方法有显著差异 (p < 0.05)")
else:
    print(f"  结论：两种方法无显著差异")

## 6. 中心极限定理演示

In [ ]:
# 从均匀分布中抽样，展示样本均值的分布

n_samples = 10000
sample_sizes = [1, 5, 30]  # 不同样本大小

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('中心极限定理演示', fontsize=14)

for ax, n in zip(axes, sample_sizes):
    sample_means = [np.mean(np.random.uniform(0, 1, n)) for _ in range(n_samples)]
    
    ax.hist(sample_means, bins=50, density=True, alpha=0.7, color='skyblue', edgecolor='black')
    
    # 添加正态分布曲线
    x = np.linspace(min(sample_means), max(sample_means), 100)
    theoretical_std = np.sqrt(1/12 / n)
    ax.plot(x, stats.norm.pdf(x, 0.5, theoretical_std), 'r-', linewidth=2)
    
    ax.set_title(f'样本大小 n = {n}')
    ax.set_xlabel('样本均值')
    ax.set_ylabel('密度')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/admin/.openclaw/workspace/DataScience_Math_Python/notebooks/clt_demo.png', dpi=150)
plt.show()

print("随着样本大小增加，样本均值的分布趋近正态分布 ✓")

## 7. 练习

In [ ]:
# 练习 1：正态分布应用
# 考试成绩服从 N(75, 100) 分布

exam_dist = stats.norm(75, 10)  # μ=75, σ=10

# 1. 成绩高于 85 分的概率
prob_above_85 = 1 - exam_dist.cdf(85)
print(f"练习 1:")
print(f"  1. P(X > 85) = {prob_above_85:.4f}")

# 2. 成绩在 65-85 分之间的概率
prob_65_85 = exam_dist.cdf(85) - exam_dist.cdf(65)
print(f"  2. P(65 ≤ X ≤ 85) = {prob_65_85:.4f}")

# 3. 前 10% 学生的最低分数
top_10_threshold = exam_dist.ppf(0.9)
print(f"  3. 前 10% 最低分数 = {top_10_threshold:.2f} 分")

In [ ]:
# 练习 2：卡方检验
# 检验骰子是否公平

# 观察频数（投掷 60 次）
observed = [8, 12, 9, 11, 10, 10]

# 期望频数（公平骰子）
expected = [10, 10, 10, 10, 10, 10]

chi2_stat, p_value_chi2 = stats.chisquare(observed, expected)

print("练习 2：卡方检验 - 骰子公平性")
print(f"  观察频数：{observed}")
print(f"  期望频数：{expected}")
print(f"  卡方统计量：{chi2_stat:.4f}")
print(f"  p 值：{p_value_chi2:.4f}")

if p_value_chi2 > 0.05:
    print(f"  结论：没有证据表明骰子不公平 (p > 0.05)")
else:
    print(f"  结论：骰子可能不公平 (p < 0.05)")

## 总结

本 Notebook 涵盖了：
- ✅ 常见概率分布（二项分布、正态分布）
- ✅ 描述统计（均值、中位数、标准差、IQR）
- ✅ 贝叶斯定理应用
- ✅ 假设检验（t 检验、卡方检验）
- ✅ 中心极限定理演示